# When Finance Deepens, Does Growth Follow? — Replication Notebook

This notebook reproduces the frozen global-panel analysis from the companion repository. It uses the archived IMF Financial Development Index, World Development Indicators, and Penn World Table 11.0 extracts; it does not retrieve live data by default.

The source universe contains 7,686 country-year observations for 183 economies (1980–2021). The complete-case estimation sample contains 3,624 observations for 135 countries (1992–2021). The threshold bootstrap uses 1,999 replications.

## 1. Load the replication repository

Upload the companion `Finance_Growth_Replication.zip` archive below. If the repository has already been cloned into the Colab runtime, set `ROOT` to that folder and skip the upload cell.

In [ ]:
import os, sys, zipfile, shutil, subprocess
from pathlib import Path
from google.colab import files

WORK = Path('/content/finance_growth_work')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
uploaded = files.upload()
zips = [name for name in uploaded if name.lower().endswith('.zip')]
if not zips:
    raise RuntimeError('Upload Finance_Growth_Replication.zip')
with zipfile.ZipFile(zips[0]) as zf:
    zf.extractall(WORK / 'extract')
candidates = [p for p in (WORK/'extract', *list((WORK/'extract').glob('*'))) if (p/'src'/'replicate_finance_growth.py').exists()]
if not candidates:
    raise RuntimeError('Could not locate src/replicate_finance_growth.py in the uploaded archive.')
ROOT = candidates[0]
os.chdir(ROOT)
print('Replication root:', ROOT)
print('Frozen input rows:', sum(1 for _ in open(ROOT/'data'/'raw_panel.csv', encoding='utf-8')) - 1)

## 2. Install the declared environment

In [ ]:
%pip install -r requirements.txt


## 3. Run the complete econometric and forecasting pipeline

The deterministic script regenerates the estimation tables, 1,999-draw threshold bootstrap, nine forecasting models, figures, and manifest. Allow the Colab runtime to finish before proceeding.

In [ ]:
subprocess.run([sys.executable, 'src/replicate_finance_growth.py'], check=True)


## 4. Run the additional robustness and validation audits

These scripts refresh the inflation-outlier robustness table, country-specific-trend results, corrected inference outputs, figure labels, independent Driscoll–Kraay cross-check, and archive validation.

In [ ]:
subprocess.run([sys.executable, 'src/audit_inflation_robustness.py'], check=True)
subprocess.run([sys.executable, 'src/audit_country_trends.py'], check=True)
subprocess.run([sys.executable, 'src/update_corrected_outputs.py'], check=True)
subprocess.run([sys.executable, 'src/update_figures_from_outputs.py'], check=True)
subprocess.run([sys.executable, 'src/dk_crosscheck_statsmodels.py'], check=True)
subprocess.run([sys.executable, 'src/validate_archive.py'], check=True)


## 5. Run the integrity tests

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## 6. Export the refreshed replication archive

The resulting ZIP contains the complete repository contents at its root, including data, source, tests, tables, figures, and the run manifest.

In [ ]:
OUT_ZIP = Path('/content/Finance_Growth_Replication_Results.zip')
if OUT_ZIP.exists(): OUT_ZIP.unlink()
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in ROOT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and '.pytest_cache' not in path.parts:
            zf.write(path, path.relative_to(ROOT))
files.download(str(OUT_ZIP))